In [7]:
import numpy as np
import pandas as pd
import string

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ami05\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [8]:
df = pd.read_csv("Preprocessed Fake Reviews Detection Dataset.csv")

df.head()

,Unnamed: 0,category,rating,label,text_
0,0,Home_and_Kitchen_5,5,1,love well made sturdi comfort i love veri pretti
1,1,Home_and_Kitchen_5,5,1,love great upgrad origin i 've mine coupl year
2,2,Home_and_Kitchen_5,5,1,thi pillow save back i love look feel pillow
3,3,Home_and_Kitchen_5,1,1,miss inform use great product price i
4,4,Home_and_Kitchen_5,5,1,veri nice set good qualiti we set two month


In [6]:
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.5 MB 12.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 12.4 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
df.drop('Unnamed: 0', axis=1, inplace=True)

df.dropna(inplace=True)

df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5,1,love well made sturdi comfort i love veri pretti
1,Home_and_Kitchen_5,5,1,love great upgrad origin i 've mine coupl year
2,Home_and_Kitchen_5,5,1,thi pillow save back i love look feel pillow
3,Home_and_Kitchen_5,1,1,miss inform use great product price i
4,Home_and_Kitchen_5,5,1,veri nice set good qualiti we set two month


In [11]:
def text_process(text):

    
    nopunc = [char for char in text if char not in string.punctuation]
    nopunc = ''.join(nopunc)

    
    return [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text_"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [13]:
models = {

    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "SVM": LinearSVC(),
    "Logistic Regression": LogisticRegression(max_iter=200)

}

In [15]:
CountVectorizer(max_features=5000)

CountVectorizer(max_features=5000)

In [16]:
pipeline = Pipeline([
    ('bow', CountVectorizer(analyzer=text_process, max_features=5000)),
    ('tfidf', TfidfTransformer()),
    ('classifier', model)
])

In [17]:
results = {}

for name, model in models.items():

    pipeline = Pipeline([
        ('bow', CountVectorizer(analyzer=text_process, max_features=5000)),
        ('tfidf', TfidfTransformer()),
        ('classifier', model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    results[name] = accuracy

    print("\nMODEL:", name)
    print("Accuracy:", accuracy)
    print(classification_report(y_test, predictions))


MODEL: Naive Bayes
Accuracy: 0.8460492147891678
              precision    recall  f1-score   support

           0       0.87      0.82      0.84      4069
           1       0.83      0.87      0.85      4018

    accuracy                           0.85      8087
   macro avg       0.85      0.85      0.85      8087
weighted avg       0.85      0.85      0.85      8087


MODEL: Random Forest
Accuracy: 0.8444416965500186
              precision    recall  f1-score   support

           0       0.86      0.82      0.84      4069
           1       0.83      0.87      0.85      4018

    accuracy                           0.84      8087
   macro avg       0.85      0.84      0.84      8087
weighted avg       0.85      0.84      0.84      8087


MODEL: Decision Tree
Accuracy: 0.7380981822678373
              precision    recall  f1-score   support

           0       0.74      0.73      0.74      4069
           1       0.73      0.75      0.74      4018

    accuracy                   

In [18]:
print("\nFINAL MODEL PERFORMANCE")

for model, acc in results.items():
    print(model, ":", round(acc*100,2), "%")


FINAL MODEL PERFORMANCE
Naive Bayes : 84.6 %
Random Forest : 84.44 %
Decision Tree : 73.81 %
KNN : 65.83 %
SVM : 87.0 %
Logistic Regression : 86.42 %


In [19]:
import joblib

joblib.dump(pipeline, "fake_review_model2.pkl")

['fake_review_model2.pkl']